# Leader-Mechanism Robustness Sensitivity Analysis

This notebook analyzes the thesis robustness experiment stored in `outputs/sensitivity_analysis/all_main`. The central quantity is the matched effect relative to a no-leader control with the same topology and seed.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'main.py').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Sensitivity_Analysis.plots import plot_core_effects, plot_share_gradient

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'sensitivity_analysis' / 'all_main'
assert OUTPUT_DIR.exists(), f'Run the formal sensitivity experiment first: {OUTPUT_DIR}'


In [ ]:
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
raw_df = pd.read_csv(OUTPUT_DIR / 'raw_results.csv')
effects_df = pd.read_csv(OUTPUT_DIR / 'matched_effects.csv')
summary_df = pd.read_csv(OUTPUT_DIR / 'summary_effects.csv')
control_df = pd.read_csv(OUTPUT_DIR / 'control_summary.csv')
checks_df = pd.read_csv(OUTPUT_DIR / 'robustness_checks.csv')

display(manifest)
print(f"Completed runs: {len(raw_df)} / {manifest['planned_run_count']}")
print(f"Matched leader effects: {len(effects_df)}")
display(checks_df.head())


## Robustness Check Summary

A result marked `preserved` retains the thesis-expected ordering or direction under the relevant parameter perturbation. For the benchmark study, CI checks additionally require the 95% interval for the matched effect to exclude zero. For share gradients, strict adjacent-step monotonicity is shown together with the broader 1%-to-5% endpoint increase.

In [ ]:
check_columns = [
    'directional_order', 'directional_ci_support',
    'one_sided_extremity', 'extremity_ci_support',
    'one_sided_homophily', 'homophily_ci_support',
    'balanced_direction_weaker',
    'directional_endpoint_increase', 'extremity_endpoint_increase', 'homophily_endpoint_increase',
]
check_summary = []
for scope, group in checks_df.groupby('check_scope'):
    for column in check_columns:
        applicable = group[group[column] != 'not_applicable']
        if applicable.empty:
            continue
        check_summary.append({
            'scope': scope,
            'criterion': column,
            'preserved': int((applicable[column] == 'preserved').sum()),
            'applicable': int(len(applicable)),
            'preserved_share': float((applicable[column] == 'preserved').mean()),
        })
check_summary_df = pd.DataFrame(check_summary)
display(check_summary_df)
display(checks_df.sort_values(['check_scope', 'topology', 'perturbation_id', 'leader_mode'], na_position='last'))


## Core Robustness at the 3% Benchmark

The forest plot treats each parameter perturbation as an independent robustness condition and reports its matched effect with a 95% confidence interval.

In [ ]:
core_df = summary_df[summary_df['study_name'] == 'core_robustness'].copy()
core_table = core_df[[
    'topology', 'perturbation_id', 'leader_mode',
    'delta_final_mean_opinion_mean', 'delta_final_mean_opinion_ci_low', 'delta_final_mean_opinion_ci_high',
    'delta_extremist_ratio_mean', 'delta_homophily_ratio_mean', 'delta_content_balance_mean',
]].sort_values(['topology', 'perturbation_id', 'leader_mode'])
display(core_table)

fig = plot_core_effects(summary_df)
plt.show()


## Leader Share Gradient Robustness

The annotated heatmap makes the overall 1%-to-5% increase visible while also revealing local saturation between 3% and 5%, especially for homophily.

In [ ]:
gradient_df = summary_df[summary_df['study_name'] == 'share_gradient'].copy()
gradient_table = gradient_df[
    gradient_df['leader_mode'].isin(['positive', 'negative'])
][[
    'topology', 'perturbation_id', 'leader_mode', 'leader_share',
    'delta_final_mean_opinion_mean', 'delta_extremist_ratio_mean', 'delta_homophily_ratio_mean',
]].sort_values(['topology', 'perturbation_id', 'leader_mode', 'leader_share'])
display(gradient_table)

fig = plot_share_gradient(summary_df)
plt.show()


## Thesis-Facing Tables

These exports provide compact tables for writing the sensitivity-analysis section after the formal run is complete.

In [ ]:
tables_dir = OUTPUT_DIR / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)
check_summary_df.to_csv(tables_dir / 'sensitivity_criterion_summary.csv', index=False)
core_table.to_csv(tables_dir / 'sensitivity_core_effects.csv', index=False)
gradient_table.to_csv(tables_dir / 'sensitivity_share_gradient_effects.csv', index=False)
print(tables_dir)
